# Sentiment Analysis on Amazon Alexa Reviews
### Using TF-IDF Vectorization + Linear SVM with Scikit-learn Pipeline

>

## Project Overview

This notebook walks through building a **binary sentiment classifier** that predicts whether an Amazon Alexa product review is **Positive** or **Negative**.

We will go through the full Machine Learning workflow:

1. **Data Loading**: Load the Amazon Alexa reviews dataset
2. **Data Pre-Processing**: Clean, transform and prepare the data for ML
3. **Feature Engineering**: Convert raw text into numerical features using TF-IDF
4. **Model Training**: Train a Linear SVM classifier using an sklearn Pipeline
5. **Model Evaluation**: Measure performance using standard classification metrics
6. **Model Serialization**: Save the trained pipeline for future use
7. **Inference**: Use the trained model to predict sentiments on new reviews

## Dataset

The dataset used is **`amazon_alexa.tsv`** — a tab-separated file containing verified customer reviews of Amazon Alexa devices. It has the following columns:

| Column | Description |
|---|---|
| `rating` | Star rating given by the customer (1 to 5) |
| `date` | Date of the review |
| `variation` | Product variant (e.g., Charcoal Fabric, Walnut Finish) |
| `verified_reviews` | The actual text of the customer review |
| `feedback` | Binary label — 1 for positive, 0 for negative |

## Tech Stack

| Library | Purpose |
|---|---|
| `pandas` | Data manipulation and analysis |
| `numpy` | Numerical operations |
| `spaCy` | NLP tokenization and text preprocessing |
| `scikit-learn` | TF-IDF vectorization, model training, and evaluation |
| `joblib` | Model serialization (saving/loading) |



>    
>
>



## 1. Import Core Libraries

We start by importing the two foundational libraries used throughout this project:

- **`numpy`**: Provides support for large multi-dimensional arrays and mathematical functions. Frequently used under the hood by pandas and scikit-learn.
- **`pandas`**: The primary tool for loading, inspecting, and manipulating tabular data (like our `.tsv` dataset).

In [40]:
!pip install imbalanced-learn

In [1]:
import numpy as np
import pandas as pd

## 2. Mount Google Drive

Since this notebook runs on **Google Colab**, we need to mount Google Drive to access project files stored there — specifically:
- The **dataset** (`amazon_alexa.tsv`)
- The **custom tokenizer** (`input_tokenizer.py`)

> **Note for newcomers:** Google Colab runs in a cloud VM that resets every session. Mounting your Drive gives the notebook persistent access to your files without re-uploading them each time.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Navigate to the Project Directory

We use the `%cd` magic command to change the working directory to the project folder inside Google Drive. This ensures that all subsequent file reads (dataset, tokenizer) are resolved relative to this path.

The `!ls` command then lists all files in the directory — useful to confirm that the required files (`input_tokenizer.py`, `amazon_alexa.tsv`) are present.

In [3]:
%cd /content/drive/My Drive/Colab Projects/1- Sentiment Analysis
!ls

/content/drive/My Drive/Colab Projects/1- Sentiment Analysis
amazon_alexa.tsv	     __pycache__
c2_Predicted_Sentiments.tsv  SentimentAnalysis_Model_Pipeline.pkl
input_tokenizer.py


## 3. Load the Dataset

The dataset is stored as a **TSV (Tab-Separated Values)** file. Unlike a standard CSV (comma-separated), columns in this file are separated by tab characters (`\t`). We use `pandas.read_csv()` with `sep='\t'` to handle this correctly.

After loading, we call `.head()` to preview the first 5 rows and confirm the data loaded correctly.

In [4]:
# Load the dataset from the TSV file
# sep='\t' tells pandas that columns are separated by tabs, not commas
alexa_data = pd.read_csv('amazon_alexa.tsv', sep='\t')

In [5]:
# Preview the first 5 rows to understand the structure of the data
alexa_data.head()

,rating,date,variation,verified_reviews,feedback
0,5,31-Jul-18,Charcoal Fabric,Love my Echo!,1
1,5,31-Jul-18,Charcoal Fabric,Loved it!,1
2,4,31-Jul-18,Walnut Finish,"Sometimes while playing a game, you can answer...",1
3,5,31-Jul-18,Charcoal Fabric,I have had a lot of fun with this thing. My 4 ...,1
4,5,31-Jul-18,Charcoal Fabric,Music,1


## 4. Data Pre-Processing

Raw data is rarely in the right format for training a machine learning model. In this section, we perform several essential pre-processing steps:

1. **Column Selection**: Keep only the two relevant columns: `verified_reviews` (the input text) and `rating` (which we'll convert into a sentiment label)
2. **Label Engineering**: Convert the 1–5 star `rating` into a binary sentiment label (0 = Negative, 1 = Positive)
3. **Missing Value Handling**: Remove any rows where the review text is missing

### 4.1 Select Relevant Columns

In [6]:
# Select only the columns needed for modeling
# 'verified_reviews' is our input (X) and 'feedback' will be our target label (y)
dataset = alexa_data[['verified_reviews', 'feedback']].copy()

# Rename columns to cleaner, more descriptive names
dataset.columns = ['Review', 'Sentiment']

In [7]:
# Verify the result
dataset.head()

,Review,Sentiment
0,Love my Echo!,1
1,Loved it!,1
2,"Sometimes while playing a game, you can answer...",1
3,I have had a lot of fun with this thing. My 4 ...,1
4,Music,1


### 4.2 Check Class Distribution

Before training, it's important to understand how the two sentiment classes are distributed in the dataset. A heavily **imbalanced dataset** (e.g., 95% positive, 5% negative) can cause a model to be biased — it may learn to always predict the majority class and still appear to have high accuracy.

Checking this distribution helps us decide whether we need resampling techniques (like oversampling the minority class).

In [8]:
# Count the number of reviews in each sentiment class
# This helps identify class imbalance before training
dataset['Sentiment'].value_counts()

,count
Sentiment,
1,2893
0,257


### 4.4 Handle Missing Values

Machine learning models cannot process `NaN` (null) values. We first check how many null values exist in each column, then drop any rows where the review text is missing.

Since the `Sentiment` column was derived from the numeric `rating` column (which has no nulls), only the `Review` column may contain missing values.

In [9]:
# Check for null values in each column
dataset.isnull().sum()

,0
Review,1
Sentiment,0


In [10]:
# Drop the single row where the Review text is null
# axis=0 (default) means we drop the entire row, not a column
dataset = dataset.dropna()

## 5. Data Transformation

### 5.1 Separate Features and Target

In supervised machine learning, we always separate the dataset into:
- **`X` (features / input)**: the data the model uses to learn patterns. Here, `X` is the review text.
- **`y` (target / label)**: what the model tries to predict. Here, `y` is the binary sentiment (0 or 1).

In [11]:
# x = input features (the raw review text)
# y = target labels (0 = Negative, 1 = Positive)
x = dataset['Review']
y = dataset['Sentiment']

### 5.2 Load the Custom Spacy Tokenizer

Before feeding text into the TF-IDF vectorizer, we need to **tokenize** and **normalize** the raw review text. We use a custom `SpacyTextPreprocessor` class defined in `input_tokenizer.py`.

**What does this tokenizer do?**
Under the hood, it uses the **spaCy** NLP library to:
1. **Tokenize**: split the text into individual words/tokens
2. **Lemmatize**: reduce words to their base/dictionary form (e.g., *"running"* → *"run"*, *"best"* → *"good"*)
3. **Remove stopwords**: filter out common, low-information words like *"the"*, *"is"*, *"and"*
4. **Remove punctuation**: strip non-alphabetic characters

This normalization ensures that the model focuses on the **meaningful content words** in each review rather than noise.

> **Example:** The sentence *"Those were the best days of my life!"* is reduced to `['good', 'day', 'life']` after lemmatization and stopword removal.

In [13]:
# Import the custom SpaCy-based text preprocessor
# This class is defined in input_tokenizer.py in the same project directory
from input_tokenizer import SpacyTextPreprocessor

In [14]:
# Instantiate the tokenizer
token = SpacyTextPreprocessor()

In [15]:
# Quick sanity check: test the tokenizer on a sample sentence
# Expected output: meaningful lemmatized tokens with stopwords removed
token.tokenize("Those were the best days of my life!")

['good', 'day', 'life']

## 6. Feature Engineering (TF-IDF Vectorization)

Machine learning algorithms work with **numbers**, not raw text. We need to convert the review text into a numerical representation that captures the importance of each word.

**TF-IDF (Term Frequency–Inverse Document Frequency)** is one of the most effective and widely used techniques for this. It assigns a numerical weight to each word in a document based on two factors:

<br>

| Component | Meaning |
|---|---|
| **Term Frequency (TF)** | How often a word appears in a specific review |
| **Inverse Document Frequency (IDF)** | How rare a word is across all reviews |

<br>

**Why does IDF matter?** A word like *"amazon"* might appear in many reviews but carries little discriminatory power. IDF down-weights such frequent words and up-weights rare, meaningful ones like *"terrible"* or *"excellent"*.

Here, we initialize `TfidfVectorizer` with our **custom spaCy tokenizer** (`token.tokenize`) instead of the default regex-based tokenizer. This ensures the preprocessing (lemmatization, stopword removal) we defined earlier is applied consistently during vectorization.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [23]:
# Initialize TF-IDF Vectorizer with our custom SpaCy tokenizer
# By passing tokenizer=token.tokenize, we override the default tokenizer
# and ensure spaCy's lemmatization + stopword removal is applied

# ngram_range=(1,2) captures both single words AND two-word phrases
# This helps catch patterns like "not good", "really bad", "works well"
tfidf = TfidfVectorizer(
    tokenizer=token.tokenize,
    ngram_range=(1, 2),
    min_df=2,           # ignore very rare terms (noise reduction)
    sublinear_tf=True   # apply log normalization to term frequencies
)

## 7. Model Training

### 7.1 Train / Test Split

We split the dataset into two subsets:
- **Training set (80%)**: used to train (fit) the model
- **Test set (20%)**: held out and used only for final evaluation

**Key parameters explained:**
<br>

| Parameter | Value | Explanation |
|---|---|---|
| `test_size` | `0.2` | 20% of data goes to the test set |
| `stratify` | `dataset.Sentiment` | Ensures both splits maintain the same class ratio as the full dataset (important when classes are imbalanced) |
| `random_state` | `0` | Sets a fixed seed so the split is reproducible across runs |

<br>

> **Why use `stratify`?** Without it, the random split might put most negative reviews (the minority class) in the training set, leaving the test set with very few negatives — making evaluation misleading.

In [24]:
from sklearn.model_selection import train_test_split

# Split into 80% training and 20% testing
# stratify ensures class proportions are preserved in both splits
# random_state ensures reproducibility
x_train, x_test, y_train, y_test = train_test_split(
    x, y,
    test_size=0.2,
    stratify=dataset.Sentiment,
    random_state=0
)

In [25]:
# Verify the sizes of the training and test sets
x_train.shape, x_test.shape

((2519,), (630,))

### 7.2 Build the Scikit-learn Pipeline

A **scikit-learn `Pipeline`** chains multiple processing steps into a single, cohesive object. Instead of manually calling `fit()` and `transform()` on each step, the pipeline handles everything automatically in the correct order.

**Our pipeline has two steps:**

```
Raw Text  →  [Step 1: TF-IDF Vectorizer]  →  Numerical Matrix  →  [Step 2: LinearSVC]  →  Prediction
```

**Why use a Pipeline?**
- **Prevents data leakage**: The vectorizer learns vocabulary only from training data and applies it (without re-learning) to the test data. If we vectorized the whole dataset first, test data statistics would leak into training.
- **Cleaner code**: One `fit()` call trains the entire workflow end-to-end.
- **Easy deployment**: The entire pipeline (preprocessing + model) can be saved as a single object and used directly for inference.

**Why Linear SVM (`LinearSVC`)?**
Linear SVMs are excellent for **high-dimensional sparse text data** (like TF-IDF vectors). They are fast to train, memory-efficient, and consistently achieve strong performance on text classification tasks.

In [42]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

In [27]:
# Instantiate the Linear SVM classifier

# class_weight='balanced' automatically compensates for the 6.7:1 imbalance
# It gives negative reviews more weight during training
classifier = LinearSVC(class_weight='balanced', C=1.0, max_iter=2000)

In [43]:
# Build the pipeline with two sequential steps:
#   Step 1 ('tfidf'): Convert raw review text into TF-IDF feature vectors
#   Step 2 ('clf'):   Train the Linear SVM classifier on those feature vectors
pipeline = ImbPipeline([
    ('tfidf', TfidfVectorizer(
        tokenizer=token.tokenize,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ('smote', SMOTE(random_state=0)),
    ('clf', LinearSVC(class_weight='balanced', C=1.0, max_iter=2000))
])

### 7.3 Train the Pipeline

Calling `pipeline.fit(x_train, y_train)` triggers the following sequence:
1. The **TF-IDF vectorizer** learns the vocabulary from `x_train` and transforms it into a numeric matrix
2. The **LinearSVC classifier** is trained on that numeric matrix and the corresponding labels `y_train`

> **Note:** You may see a `UserWarning` about `token_pattern` being ignored. This is expected, it's scikit-learn informing us that the default regex token pattern is being replaced by our custom spaCy tokenizer. This warning can be safely disregarded.

In [ ]:
# Train the full pipeline (vectorization + classification) on the training data
# This single call handles both fitting the TF-IDF and training the SVM
pipeline.fit(x_train, y_train)

## 8. Model Evaluation

After training, we evaluate the model on the **held-out test set** — data the model has never seen before. This gives an unbiased estimate of real-world performance.

We use three standard evaluation tools:

### 8.1 Generate Predictions

We call `pipeline.predict(x_test)` to apply the trained pipeline to the test reviews. Internally, this:
1. Applies the already-fitted TF-IDF vectorizer to transform the test text
2. Passes the resulting vectors to the trained LinearSVC to produce predictions

In [45]:
# Generate predictions on the unseen test set
# The pipeline applies the same TF-IDF transformation learned during training
y_pred = pipeline.predict(x_test)

In [46]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [47]:
y_pred = pipeline.predict(x_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Negative', 'Positive']))

# Pay attention to: Negative recall (how many negatives it correctly catches)

Accuracy: 0.9063492063492063

Classification Report:
              precision    recall  f1-score   support

    Negative       0.45      0.65      0.53        51
    Positive       0.97      0.93      0.95       579

    accuracy                           0.91       630
   macro avg       0.71      0.79      0.74       630
weighted avg       0.93      0.91      0.91       630



### 8.2 Confusion Matrix

A **confusion matrix** is a 2×2 table that summarizes the model's predictions vs the actual labels:

```
                  Predicted Negative   Predicted Positive
Actual Negative   True Negative (TN)   False Positive (FP)
Actual Positive   False Negative (FN)  True Positive (TP)
```

- **True Positives (TP)**: Correctly predicted positive reviews
- **True Negatives (TN)**: Correctly predicted negative reviews
- **False Positives (FP)**: Negative reviews incorrectly predicted as positive (Type I Error)
- **False Negatives (FN)**: Positive reviews incorrectly predicted as negative (Type II Error)

In [48]:
# Confusion matrix: rows = actual labels, columns = predicted labels
# Format: [[TN, FP], [FN, TP]]
confusion_matrix(y_test, y_pred)

array([[ 33,  18],
       [ 41, 538]])

### 8.3 Classification Report

The **classification report** provides a richer picture of model performance with the following per-class metrics:

| Metric | Formula | What It Measures |
|---|---|---|
| **Precision** | TP / (TP + FP) | Of all reviews predicted positive, how many actually were? |
| **Recall** | TP / (TP + FN) | Of all actually positive reviews, how many did we catch? |
| **F1-Score** | 2 × (P × R) / (P + R) | Harmonic mean of precision and recall — useful for imbalanced datasets |
| **Support** | — | Number of actual instances of each class in the test set |

> **Tip:** Because our dataset is imbalanced (many more positives than negatives), pay more attention to the **F1-Score** and **per-class metrics** than to overall accuracy alone.

In [49]:
# Print a full classification report with per-class precision, recall, and F1-score
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.45      0.65      0.53        51
           1       0.97      0.93      0.95       579

    accuracy                           0.91       630
   macro avg       0.71      0.79      0.74       630
weighted avg       0.93      0.91      0.91       630



# 9. Model Serialization

Once satisfied with model performance, we **serialize** (save) the trained pipeline to disk using `joblib`. This lets us:
- Avoid retraining the model every time we want to make a prediction
- Load the model directly in a production API or separate script

**Why `joblib` over `pickle`?**
`joblib` is the recommended serialization tool for scikit-learn objects. It is more efficient than Python's built-in `pickle` for objects containing large NumPy arrays (like our fitted TF-IDF matrix), and it supports compressed saving.

The saved `.pkl` file contains the **entire pipeline** — the fitted TF-IDF vocabulary and the trained SVM weights — as a single reusable object.

In [50]:
import joblib

# Serialize and save the entire pipeline (TF-IDF + SVM) to a .pkl file
# This file can be loaded later for inference without retraining
joblib.dump(pipeline, 'SentimentAnalysis_Model_Pipeline.pkl')

['SentimentAnalysis_Model_Pipeline.pkl']

## 10. Inference (Predicting Sentiment on New Reviews)

Now that the model is trained and saved, we can use it to classify **new, unseen reviews**. The pipeline handles all preprocessing internally, we simply pass in the raw text string.

### 10.1 Single Prediction (Simple Method)

For a quick one-off prediction, we pass a single review as a list (the vectorizer expects an iterable) and interpret the result.

In [58]:
# Pass a new review as a single-element list to the trained pipeline
# The pipeline applies TF-IDF transformation and then predicts the label
prediction = pipeline.predict(["Its ok"])

# Interpret and print the result
if prediction == 1:
    print("Result: This review is Positive")
else:
    print(prediction)
    print("Result: This review is Negative")

[0]
Result: This review is Negative


### 10.2 Interactive Batch Prediction (Loop Method)

For a more interactive experience, this loop continuously prompts the user to enter a review and immediately returns the sentiment prediction. Entering **`'skip'`** exits the loop.

All entered reviews and their predicted sentiments are collected into lists, which are later saved to a TSV file for record-keeping.

> **Note:** This interactive `input()` loop is designed for notebook use. In a production API, this would be replaced by an HTTP request handler.

In [ ]:
# Lists to collect all reviewed inputs and their predicted sentiments
new_review = []
pred_sentiment = []

while True:
    # Prompt the user to enter a new Alexa product review
    review = input("Please type an Alexa review (or type 'skip' to exit): ")

    # Exit condition: type 'skip' to stop the loop
    if review == 'skip':
        print("See you soon!")
        break
    else:
        # Predict the sentiment of the entered review
        prediction = pipeline.predict([review])

        # Interpret and display the result
        if prediction == 1:
            result = 'Positive'
            print("Result: This review is Positive\n")
        else:
            result = 'Negative'
            print("Result: This review is Negative\n")

    # Append the review and its result to the tracking lists
    new_review.append(review)
    pred_sentiment.append(result)

Please type an Alexa review (or type 'skip' to exit): This product is too good
Result: This review is Positive

Please type an Alexa review (or type 'skip' to exit): skip
See you soon!


### 10.3 Save Prediction Results to File

We compile the reviews and their predicted sentiments into a `DataFrame` and export it as a TSV file. This provides an auditable record of all predictions made during this session, and can be used for manual review or further analysis.

In [ ]:
# Compile the reviews and predictions into a DataFrame
Results_Summary = pd.DataFrame({
    'New Review': new_review,
    'Sentiment': pred_sentiment
})

# Export to a TSV file (tab-separated, UTF-8 encoded, without row indices)
Results_Summary.to_csv(
    "./c2_Predicted_Sentiments.tsv",
    sep='\t',
    encoding='UTF-8',
    index=False
)

# Display the results summary
Results_Summary

,New Review,Sentiment
0,This product is too good,Positive


## 11. Load Saved Model from Disk (Optional)

This section demonstrates how to **reload the saved pipeline** from the `.pkl` file and use it directly for inference, without needing to retrain the model.

This is the standard pattern for deploying a trained model in a production environment: train once, save, and load on demand.

In [ ]:
# Uncomment to load the saved pipeline from disk
# model = joblib.load('SentimentAnalysis_Model_Pipeline.pkl')

In [ ]:
# Define a new review to test the loaded model
# new_review = ["bad product"]

In [ ]:
# Run prediction using the loaded model
# Returns 0 (Negative) or 1 (Positive)
# model.predict(new_review)[0]